# Feature engineering

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis

df = pd.read_csv('./data/all.csv')

# Replace NaNs with new class
df['term_id'] = df['term_id'].fillna(-1)  # or any unique sentinel value

# Ensure datetime is parsed
df['tr_datetime'] = pd.to_datetime(df['tr_datetime'], errors='coerce')
assert np.issubdtype(df['tr_datetime'].dtype, np.datetime64), "tr_datetime must be parsed"

# ─────────────────────────────────────
# Feature Engineering Per Customer
# ─────────────────────────────────────

# 1. Basic transaction stats + counts of unique categorical features
tx_stats = df.groupby('customer_id').agg({
    'amount': ['count', 'sum', 'mean', 'std', 'min', 'max'],
    'term_id': pd.Series.nunique,
    'mcc_code': pd.Series.nunique,
    'tr_type': pd.Series.nunique
})

# Flatten multiindex columns
tx_stats.columns = [f'base_{k}_{stat}' for k, stat in tx_stats.columns]
tx_stats.reset_index(inplace=True)

# 2. Positive vs negative transaction breakdown
df['is_positive'] = (df['amount'] > 0).astype(int)
df['is_negative'] = (df['amount'] < 0).astype(int)
pos_neg_stats = df.groupby('customer_id').agg({
    'is_positive': 'mean',
    'is_negative': 'mean'
}).rename(columns={
    'is_positive': 'share_positive_txn',
    'is_negative': 'share_negative_txn'
}).reset_index()

# 3. Enhanced amount aggregations
df['amount_positive'] = df['amount'].where(df['amount'] > 0, 0)
df['amount_negative'] = df['amount'].where(df['amount'] < 0, 0)

agg_funcs = {
    'amount': ['median',
               lambda x: np.percentile(x, 25),
               lambda x: np.percentile(x, 75),
               skew,
               kurtosis
               ],
    'amount_positive': ['sum', 'count'],
    'amount_negative': ['sum', 'count']
}

amount_stats = df.groupby('customer_id').agg(agg_funcs)

# Rename columns for clarity
amount_stats.columns = [
    'amount_median',
    'amount_pct25',
    'amount_pct75',
    'amount_skew',
    'amount_kurtosis',
    'amount_positive_sum',
    'amount_positive_count',
    'amount_negative_sum',
    'amount_negative_count'
]

# Calculate ratios and shares
amount_stats['amount_pos_neg_sum_ratio'] = (
    amount_stats['amount_positive_sum'].abs() / (amount_stats['amount_negative_sum'].abs() + 1e-9)
)
amount_stats['amount_pos_count_share'] = (
    amount_stats['amount_positive_count'] / (amount_stats['amount_positive_count'] + amount_stats['amount_negative_count'] + 1e-9)
)

# Handle inf and NaN
amount_stats.replace([np.inf, -np.inf], 0, inplace=True)
amount_stats.fillna(0, inplace=True)
amount_stats.reset_index(inplace=True)

# 3. Temporal patterns
df['hour'] = df['tr_datetime'].dt.hour
df['minute'] = df['tr_datetime'].dt.minute
df['weekday'] = df['tr_datetime'].dt.weekday
df['day'] = (df['tr_datetime'] - pd.to_datetime("2000-01-01")).dt.days
df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)
df['days_since_last_txn'] = df.groupby('customer_id')['day'].transform(lambda x: x.max() - x)

# Add cyclic features
# Hour of day (0-23)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# Minute of hour (0-59)
df['minute'] = df['tr_datetime'].dt.minute
df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)

# Weekday (0-6)
df['weekday_sin'] = np.sin(2 * np.pi * df['weekday'] / 7)
df['weekday_cos'] = np.cos(2 * np.pi * df['weekday'] / 7)

# Day (assuming days numbered from some base, max_day is max)
max_day = df['day'].max() if 'day' in df else 365
df['day_sin'] = np.sin(2 * np.pi * df['day'] / max_day)
df['day_cos'] = np.cos(2 * np.pi * df['day'] / max_day)

df.drop(columns=['day', 'weekday', 'hour', 'minute']) # will have a lot of unnecessary noise and are redundant

temporal_stats = df.groupby('customer_id').agg({
    # 'minute': ['mean', 'std', 'min', 'max'],
    # 'weekday': ['mean', 'std', pd.Series.nunique],
    'is_weekend': 'mean',
    # 'day': ['min', 'max', pd.Series.nunique],
    'hour_sin': ['mean'],
    'hour_cos': ['mean'],
    'weekday_sin': ['mean'],
    'weekday_cos': ['mean'],
    'day_sin': ['mean'],
    'day_cos': ['mean'],
    'days_since_last_txn': 'mean'
})
temporal_stats.columns = [f'temp_{k}_{stat}' for k, stat in temporal_stats.columns]
temporal_stats.reset_index(inplace=True)

# Days between first and last txn
# temporal_stats['temp_days_active'] = temporal_stats['temp_day_max'] - temporal_stats['temp_day_min']
# temporal_stats.drop(columns=['temp_day_min', 'temp_day_max'], inplace=True)

# 4. MCC frequency
top_mcc = df['mcc_code'].value_counts().index
df_top_mcc = df[df['mcc_code'].isin(top_mcc)]
mcc_freq = pd.crosstab(df_top_mcc['customer_id'], df_top_mcc['mcc_code'])
mcc_freq.columns = [f'mcc_{c}' for c in mcc_freq.columns]
mcc_freq.reset_index(inplace=True)

# 5. Transaction type frequency
trtype_freq = pd.crosstab(df['customer_id'], df['tr_type'])
trtype_freq.columns = [f'trtype_{c}' for c in trtype_freq.columns]
trtype_freq.reset_index(inplace=True)

# 6. Transaction density
txn_density = df.groupby('customer_id')['day'].agg(['count', 'nunique'])
txn_density['txn_per_day'] = txn_density['count'] / txn_density['nunique']
txn_density = txn_density[['txn_per_day']].reset_index()

# 7. MCC codes and transaction types embeddings

import json
with open('./data/mcc2emb.json') as f:
    mcc2emb = {int(k): np.array(v) for k, v in json.load(f).items()}

with open('./data/tr_type2emb.json') as f:
    tr_type2emb = {int(k): np.array(v) for k, v in json.load(f).items()}

def add_embedding_features_to_df(df, mcc2emb, tr_type2emb):
    mcc_emb_dim = len(next(iter(mcc2emb.values())))
    tr_emb_dim = len(next(iter(tr_type2emb.values())))

    mcc_emb_matrix = np.stack([
        mcc2emb.get(code, np.zeros(mcc_emb_dim)) for code in df['mcc_code']
    ])
    tr_emb_matrix = np.stack([
        tr_type2emb.get(code, np.zeros(tr_emb_dim)) for code in df['tr_type']
    ])

    mcc_emb_df = pd.DataFrame(mcc_emb_matrix, columns=[f'mcc_emb_{i}' for i in range(mcc_emb_dim)])
    tr_emb_df = pd.DataFrame(tr_emb_matrix, columns=[f'tr_type_emb_{i}' for i in range(tr_emb_dim)])

    df = pd.concat([df.reset_index(drop=True), mcc_emb_df, tr_emb_df], axis=1)
    return df

def aggregate_embedding_features(df, mcc_emb_dim, tr_emb_dim):
    mcc_cols = [f'mcc_emb_{i}' for i in range(mcc_emb_dim)]
    tr_cols = [f'tr_type_emb_{i}' for i in range(tr_emb_dim)]

    emb_cols = mcc_cols + tr_cols
    emb_agg = df.groupby('customer_id')[emb_cols].agg(['mean'])

    # Flatten column names
    emb_agg.columns = [f'{col}_{stat}' for col, stat in emb_agg.columns]
    emb_agg.reset_index(inplace=True)
    return emb_agg

# df = add_embedding_features_to_df(df, mcc2emb, tr_type2emb)
# emb_agg = aggregate_embedding_features(df, mcc_emb_dim=10, tr_emb_dim=10)

# ─────────────────────────────────────
# Merge all features
# ─────────────────────────────────────
features = tx_stats \
    .merge(pos_neg_stats, on='customer_id', how='left') \
    .merge(amount_stats, on='customer_id', how='left') \
    .merge(temporal_stats, on='customer_id', how='left') \
    .merge(txn_density, on='customer_id', how='left') \
    .merge(mcc_freq, on='customer_id', how='left') \
    .merge(trtype_freq, on='customer_id', how='left') 
    # .merge(emb_agg, on='customer_id', how='left')

# Add target
target = df[['customer_id', 'gender']].drop_duplicates()
features = features.merge(target, on='customer_id', how='left')

# Final data
X = features.drop(columns=['customer_id', 'gender'])
y = features['gender']

In [2]:
# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import PCA

# scaler = StandardScaler()
# X_no_nans = X.fillna(0.0)

# X_scaled = scaler.fit_transform(X_no_nans)

# pca = PCA(n_components=0.95)  # keep 95% variance
# X_pca = pca.fit_transform(X_scaled)

# X_pca = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])

# len(X_pca.columns)

### Sequential data preprocessing

Reduce number of transactions per user to 1024 max

In [3]:
import pandas as pd
import numpy as np

def select_representative_transactions(df, max_total=1024, first_n=256, last_n=256, middle_n=512):
    """
    For each customer in df, select up to `max_total` transactions:
    - First `first_n`
    - Last `last_n`
    - `middle_n` transactions spread evenly from the middle segment

    Returns:
        A reduced DataFrame with selected transactions per customer
    """
    assert 'customer_id' in df.columns and 'tr_datetime' in df.columns, "Missing required columns"
    df = df.sort_values(['customer_id', 'tr_datetime'])

    selected_rows = []

    for cid, group in df.groupby('customer_id'):
        n = len(group)

        # Select first and last
        first_part = group.iloc[:first_n]
        last_part = group.iloc[-last_n:]

        # Define middle range (avoid overlap)
        start = first_n
        end = max(n - last_n, start)
        middle_len = max(end - start, 0)

        if middle_len > 0 and middle_n > 0:
            indices = np.linspace(start, end - 1, min(middle_n, middle_len), dtype=int)
            middle_part = group.iloc[indices]
        else:
            middle_part = pd.DataFrame(columns=group.columns)

        # Only concat non-empty parts to avoid future warnings
        parts = [first_part]
        if not middle_part.empty:
            parts.append(middle_part)
        parts.append(last_part)

        selected = pd.concat(parts)
        selected_rows.append(selected)

    return pd.concat(selected_rows, ignore_index=True)

# df_sequential = select_representative_transactions(df)
# df_sequential = df_sequential[["customer_id", "mcc_code", "tr_type", "amount", "term_id", "gender", "hour", "day", "weekday", "minute_sin", "minute_cos"]]
# df_sequential.head()

# Training

In [5]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, average_precision_score,
    precision_score, recall_score
)
from xgboost import XGBClassifier

# Stratified K-Fold setup
n_splits = 3
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Store scores
auc_scores = []
accuracy_scores = []
avg_precision_scores = []
precision_scores = []
recall_scores = []

best_params = {
    "n_estimators": 771,
    "max_depth": 7,
    "learning_rate": 0.013280302507448957,
    "subsample": 0.731284579500826,
    "colsample_bytree": 0.856065289978587,
    "min_child_weight": 6,
    "reg_alpha": 1.035083084562599e-05,
    "reg_lambda": 2.61278637853653e-07,
}

# Cross-validation loop
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"Fold {fold}:")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model = XGBClassifier(
        **best_params,
        eval_metric='logloss',
        n_jobs=-1,
        random_state=42
    )
    model.fit(X_train, y_train)

    y_pred_proba = model.predict_proba(X_val)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    auc = roc_auc_score(y_val, y_pred_proba)
    acc = accuracy_score(y_val, y_pred)
    ap = average_precision_score(y_val, y_pred_proba)
    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)

    auc_scores.append(auc)
    accuracy_scores.append(acc)
    avg_precision_scores.append(ap)
    precision_scores.append(precision)
    recall_scores.append(recall)

    print(f"  ROC AUC:            {auc:.4f}")
    print(f"  Accuracy:           {acc:.4f}")
    print(f"  Average Precision:  {ap:.4f}")
    print(f"  Precision:          {precision:.4f}")
    print(f"  Recall:             {recall:.4f}")

# Final results
print("\nFinal Cross-Validation Scores:")
print(f"Mean ROC AUC:           {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print(f"Mean Accuracy:          {np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}")
print(f"Mean Average Precision: {np.mean(avg_precision_scores):.4f} ± {np.std(avg_precision_scores):.4f}")
print(f"Mean Precision:         {np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}")
print(f"Mean Recall:            {np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}")

Fold 1:
  ROC AUC:            0.8730
  Accuracy:           0.7871
  Average Precision:  0.8544
  Precision:          0.7841
  Recall:             0.7157
Fold 2:
  ROC AUC:            0.8817
  Accuracy:           0.8007
  Average Precision:  0.8601
  Precision:          0.7916
  Recall:             0.7456
Fold 3:
  ROC AUC:            0.8796
  Accuracy:           0.7939
  Average Precision:  0.8565
  Precision:          0.7984
  Recall:             0.7138

Final Cross-Validation Scores:
Mean ROC AUC:           0.8781 ± 0.0037
Mean Accuracy:          0.7939 ± 0.0055
Mean Average Precision: 0.8570 ± 0.0024
Mean Precision:         0.7913 ± 0.0058
Mean Recall:            0.7250 ± 0.0145


In [6]:
model_path = f"./saved_models/xgb_fold{fold}.model"
model.save_model(model_path)
print(f"Saved model to: {model_path}")

Saved model to: ./saved_models/xgb_fold3.model


/cephfs/home/savkin/miniconda3/envs/classic_ml/lib/python3.12/site-packages/xgboost/sklearn.py:1028: UserWarning: [21:40:52] WARNING: /workspace/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)


## Optuna

In [ ]:
from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier

# XGBoost train+predict function
def train_predict_xgb(params, X_train, y_train, X_val, y_val):
    model = XGBClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        # early_stopping_rounds=50,
        verbose=False
    )
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    return y_pred_proba

In [ ]:
# CatBoost train+predict function
def train_predict_catboost(params, X_train, y_train, X_val, y_val):
    train_pool = Pool(X_train, y_train)
    val_pool = Pool(X_val, y_val)
    model = CatBoostClassifier(**params)
    model.fit(
        train_pool,
        eval_set=val_pool,
        use_best_model=True,
        verbose=False
    )
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    return y_pred_proba

def train_predict_lstm(params, X_train, y_train, X_val, y_val):
    return None

In [ ]:
df.groupby("customer_id").count()

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pandas as pd
import numpy as np

class TransactionSequenceDataset(Dataset):
    def __init__(self, df, customer_ids, max_seq_len=100):
        self.customer_ids = customer_ids
        self.max_seq_len = max_seq_len

        df = df.copy()
        self.features = ['amount', 'hour', 'day', 'weekday' 'minute_sin', 'minute_cos' 'mcc_code', 'tr_type']  # example feature set

        # Normalize continuous features
        scaler = StandardScaler()
        df['amount'] = scaler.fit_transform(df[['amount']])
        df['minute_sin'] = scaler.fit_transform(df[['amminute_sinount']])
        df['minute_cos'] = scaler.fit_transform(df[['minute_cos']])

        
        # Encode categorical features
        df['mcc_code'] = LabelEncoder().fit_transform(df['mcc_code'])
        df['tr_type'] = LabelEncoder().fit_transform(df['tr_type'])

        self.df = df

    def __len__(self):
        return len(self.customer_ids)

    def __getitem__(self, idx):
        cid = self.customer_ids[idx]
        txns = self.df[self.df['customer_id'] == cid].sort_values('tr_datetime')
        seq = txns[self.features].values[-self.max_seq_len:]
        
        # Pad sequence
        if len(seq) < self.max_seq_len:
            pad = np.zeros((self.max_seq_len - len(seq), len(self.features)))
            seq = np.vstack([pad, seq])
        
        x = torch.tensor(seq, dtype=torch.float32)
        y = txns['gender'].iloc[0]
        return x, torch.tensor(y, dtype=torch.float32)

class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, 
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.out = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        return torch.sigmoid(self.out(hn[-1]))

def train_predict_lstm(params, X_train, y_train, X_val, y_val):
    # Use global raw df
    global df

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    batch_size = 64
    max_seq_len = 512

    train_ids = X_train.index
    val_ids = X_val.index

    train_dataset = TransactionSequenceDataset(df, train_ids, max_seq_len=max_seq_len)
    val_dataset = TransactionSequenceDataset(df, val_ids, max_seq_len=max_seq_len)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    input_size = len(train_dataset[0][0][0])
    model = LSTMClassifier(input_size, 
                           hidden_size=params.get('hidden_size', 64),
                           num_layers=params.get('num_layers', 1),
                           dropout=params.get('dropout', 0.1)).to(device)

    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=params.get('learning_rate', 1e-3))

    # Train loop
    model.train()
    for epoch in range(5):  # keep short for tuning
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).squeeze()
            loss = criterion(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Eval
    model.eval()
    all_preds = []
    with torch.no_grad():
        for xb, _ in val_loader:
            xb = xb.to(device)
            preds = model(xb).squeeze().cpu().numpy()
            all_preds.extend(preds)

    return np.array(all_preds)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, average_precision_score
from xgboost import XGBClassifier
import optuna
import numpy as np
from tqdm import tqdm

X = features.drop(columns=['customer_id', 'gender'])
y = features['gender'].values

# Stratified K-Fold params
n_splits = 3
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

MODEL = "XGB"
# MODEL = "CATBOOST"
# MODEL = "LSTM"

PARAM_SPACE = {
    "XGB": {
        'n_estimators': (100, 1000, 'int'),
        'max_depth': (3, 10, 'int'),
        'learning_rate': (1e-3, 0.3, 'log'),
        'subsample': (0.6, 1.0, 'float'),
        'colsample_bytree': (0.6, 1.0, 'float'),
        'min_child_weight': (1, 10, 'int'),
        'reg_alpha': (1e-8, 10.0, 'log'),
        'reg_lambda': (1e-8, 10.0, 'log'),
    },
    "CATBOOST": {
        'iterations': (100, 1000, 'int'),
        'depth': (3, 10, 'int'),
        'learning_rate': (1e-3, 0.3, 'log'),
        'random_strength': (1e-9, 10, 'log'),
        'l2_leaf_reg': (1, 10, 'float'),
        'border_count': (32, 255, 'int'),
    },
    "LSTM": {
        # example params, fill with your own if needed
        'hidden_size': (32, 256, 'int'),
        'num_layers': (1, 3, 'int'),
        'learning_rate': (1e-4, 1e-2, 'log'),
        'dropout': (0.0, 0.5, 'float'),
    }
}

def sample_params(trial, model_name):
    params = {}
    space = PARAM_SPACE[model_name]
    for k, v in space.items():
        low, high, ptype = v
        if ptype == 'int':
            params[k] = trial.suggest_int(k, low, high)
        elif ptype == 'float':
            params[k] = trial.suggest_float(k, low, high)
        elif ptype == 'log':
            params[k] = trial.suggest_float(k, low, high, log=True)
    return params

def objective(trial):
    params = sample_params(trial, MODEL)

    # Add fixed params common to models if needed
    if MODEL == "XGB":
        params.update({
            'random_state': 42,
            'n_jobs': -1,
            'use_label_encoder': False,
            'eval_metric': 'logloss',
        })
    elif MODEL == "CATBOOST":
        params.update({
            'random_seed': 42,
            'verbose': False,
            'loss_function': 'Logloss',
            'eval_metric': 'AUC',
        })
    elif MODEL == "LSTM":
        # Add your fixed params or leave blank
        pass

    auc_scores = []

    for train_idx, val_idx in tqdm(skf.split(X, y), desc=f"Training folds {MODEL}"):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        if MODEL == "XGB":
            y_pred_proba = train_predict_xgb(params, X_train, y_train, X_val, y_val)
        elif MODEL == "CATBOOST":
            y_pred_proba = train_predict_catboost(params, X_train, y_train, X_val, y_val)
        elif MODEL == "LSTM":
            # Placeholder function, define your LSTM train/predict function
            y_pred_proba = train_predict_lstm(params, X_train, y_train, X_val, y_val)
        else:
            raise ValueError(f"Unknown MODEL: {MODEL}")

        auc = roc_auc_score(y_val, y_pred_proba)
        auc_scores.append(auc)

    return np.mean(auc_scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

print(f"Best trial ROC AUC: {study.best_value:.4f}")
print("Best hyperparameters:")
for key, val in study.best_params.items():
    print(f"{key}: {val}")